# Online Shopping Purchase Intention Prediction Using Machine Learning
**University AI & Machine Learning Assignment (Year 2 Semester 1)**

- **Primary Model**: Support Vector Machine (SVM with RBF Kernel)
- **Comparison Model**: Decision Tree Classifier
- **Dataset**: `online_shoppers_intention.csv` (12,330 rows, 18 columns)
- **Target**: `Revenue` (Binary: `False` / `True`)


### Cell 1: Environment Setup & Dataset Ingestion
This cell downloads the official dataset directly from the UCI Machine Learning repository if not already uploaded.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import urllib.request

# Download dataset if not present
dataset_filename = 'online_shoppers_intention.csv'
if not os.path.exists(dataset_filename):
    print("Downloading dataset from UCI Machine Learning Repository...")
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00468/online_shoppers_intention.csv'
    urllib.request.urlretrieve(url, dataset_filename)
    print("Download complete!")

# Load dataset
df = pd.read_csv(dataset_filename)
print(f"Dataset successfully loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


### Cell 2: Dataset Understanding & Schema Verification
Verifies columns, data types, missing values, and checks the target class imbalance.


In [ ]:
print("=== SCHEMA AUDIT & DATA TYPES ===")
print(df.info())

print("\n=== MISSING VALUES COUNT ===")
print(df.isnull().sum())

print("\n=== REVENUE CLASS DISTRIBUTION (TARGET) ===")
counts = df['Revenue'].value_counts()
pcts = df['Revenue'].value_counts(normalize=True) * 100
summary_target = pd.DataFrame({'Sessions': counts, 'Percentage (%)': pcts.round(2)})
print(summary_target)


### Cell 3: Exploratory Data Analysis (EDA) Visualizations
Generates publication-quality charts examining class imbalance, visitor behavior (`PageValues`, `BounceRates`, `ExitRates`), and monthly trends.


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Target Class Imbalance
bars = axes[0, 0].bar(['No Purchase (False)', 'Purchase (True)'], counts, color=['#3B82F6', '#10B981'], width=0.5)
axes[0, 0].set_title('1. Revenue Class Imbalance Distribution', fontweight='bold')
axes[0, 0].set_ylabel('Number of Sessions')
for bar in bars:
    yval = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2, yval + 150, f"{yval:,}", ha='center', fontweight='bold')

# 2. Monthly Revenue Distribution
month_order = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
sns.countplot(data=df, x='Month', hue='Revenue', order=month_order, ax=axes[0, 1], palette=['#3B82F6', '#10B981'])
axes[0, 1].set_title('2. Session Revenue Outcomes by Month', fontweight='bold')
axes[0, 1].set_ylabel('Number of Sessions')

# 3. PageValues (Dominant Feature) Distribution (Log scale)
sns.histplot(data=df, x='PageValues', hue='Revenue', bins=30, kde=True, ax=axes[1, 0], palette=['#3B82F6', '#10B981'], element='step')
axes[1, 0].set_title('3. PageValues Distribution (Key Predictor)', fontweight='bold')
axes[1, 0].set_yscale('log')

# 4. BounceRates vs ExitRates
sns.scatterplot(data=df, x='BounceRates', y='ExitRates', hue='Revenue', alpha=0.6, ax=axes[1, 1], palette=['#3B82F6', '#10B981'])
axes[1, 1].set_title('4. BounceRates vs ExitRates Telemetry', fontweight='bold')

plt.tight_layout()
plt.show()


### Cell 4: Correlation Heatmap for Numerical Predictors


In [ ]:
num_cols = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates',
    'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]

plt.figure(figsize=(11, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', vmin=-1, vmax=1, annot=True, fmt='.2f')
plt.title('Pearson Correlation Heatmap for Numerical Predictors', fontweight='bold')
plt.show()


### Cell 5: Preprocessing Pipelines & 80/20 Stratified Train-Test Split
- 80% Training (9,864 samples), 20% Testing (2,466 samples)
- Stratified on `Revenue` (`random_state=42`)
- The test set remains completely **unseen** during training and hyperparameter tuning to prevent data leakage.
- `StandardScaler` applied to 14 numerical features for SVM.
- `OneHotEncoder(handle_unknown='ignore')` applied to categorical features (`Month`, `VisitorType`).
- `Weekend` converted to numeric 0/1.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Prepare features and target
X = df.drop(columns=['Revenue']).copy()
X['Weekend'] = X['Weekend'].astype(int)
y = df['Revenue'].astype(int)

# Stratified 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples (Purchase rate: {y_train.mean():.4f})")
print(f"Testing set:  {X_test.shape[0]} samples (Purchase rate: {y_test.mean():.4f})")

cat_cols = ['Month', 'VisitorType']
bool_cols = ['Weekend']

# Preprocessor for SVM (StandardScaler on numerical)
preprocessor_svm = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('bool', 'passthrough', bool_cols)
    ]
)

# Preprocessor for Decision Tree (Decision Trees are scale-invariant)
preprocessor_dt = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('bool', 'passthrough', bool_cols)
    ]
)


### Cell 6: Model 1 Training & Hyperparameter Tuning — SVM (Primary Model)
- Model: `sklearn.svm.SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)`
- 5-Fold Stratified Cross-Validation maximizing **F1-score**
- Tuning $C \in [0.1, 1.0, 10.0, 100.0]$ and $\gamma \in [	ext{'scale'}, 0.01, 0.001]$


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold

svm_pipeline = Pipeline([
    ('preprocessor', preprocessor_svm),
    ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
])

svm_param_grid = {
    'classifier__C': [0.1, 1.0, 10.0, 100.0],
    'classifier__gamma': ['scale', 0.01, 0.001]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

print("Running 5-Fold Stratified GridSearchCV for SVM...")
svm_grid.fit(X_train, y_train)
best_svm = svm_grid.best_estimator_

best_svm_params = {k.replace('classifier__', ''): v for k, v in svm_grid.best_params_.items()}
print(f"Optimal SVM Hyperparameters: {best_svm_params}")
print(f"Best 5-Fold CV Mean F1-Score: {svm_grid.best_score_:.4f}")


### Cell 7: Model 2 Training & Hyperparameter Tuning — Decision Tree (Comparison Model)
- Model: `sklearn.tree.DecisionTreeClassifier(class_weight='balanced', random_state=42)`
- 5-Fold Stratified Cross-Validation maximizing **F1-score**
- Tuning `max_depth`, `min_samples_split`, `min_samples_leaf`, and `criterion`


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_pipeline = Pipeline([
    ('preprocessor', preprocessor_dt),
    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

dt_param_grid = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

print("Running 5-Fold Stratified GridSearchCV for Decision Tree...")
dt_grid.fit(X_train, y_train)
best_dt = dt_grid.best_estimator_

best_dt_params = {k.replace('classifier__', ''): v for k, v in dt_grid.best_params_.items()}
print(f"Optimal Decision Tree Hyperparameters: {best_dt_params}")
print(f"Best 5-Fold CV Mean F1-Score: {dt_grid.best_score_:.4f}")


### Cell 8: Model Evaluation on the Untouched Test Set (N=2,466)
Calculates Accuracy, Precision, Recall, F1-Score, ROC-AUC, and Confusion Matrices on the unseen test set.


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

def evaluate_pipeline(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, y_pred, digits=4, zero_division=0))
    
    return {
        'name': name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'roc_auc': auc,
        'cm': cm,
        'y_proba': y_proba
    }

svm_results = evaluate_pipeline(best_svm, X_test, y_test, name="SVM (Primary)")
dt_results = evaluate_pipeline(best_dt, X_test, y_test, name="Decision Tree (Comparison)")


### Cell 9: Evaluation Diagnostic Curves (Confusion Matrices, ROC, PR Curves)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrices
fig_cm, axes_cm = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(svm_results['cm'], annot=True, fmt='d', cmap='Blues', ax=axes_cm[0],
            xticklabels=['Pred: False', 'Pred: True'], yticklabels=['Actual: False', 'Actual: True'])
axes_cm[0].set_title(f"SVM Confusion Matrix\nAccuracy: {svm_results['accuracy']:.4f}")
axes_cm[0].set_ylabel('Actual Class')
axes_cm[0].set_xlabel('Predicted Class')

sns.heatmap(dt_results['cm'], annot=True, fmt='d', cmap='Greens', ax=axes_cm[1],
            xticklabels=['Pred: False', 'Pred: True'], yticklabels=['Actual: False', 'Actual: True'])
axes_cm[1].set_title(f"Decision Tree Confusion Matrix\nAccuracy: {dt_results['accuracy']:.4f}")
axes_cm[1].set_ylabel('Actual Class')
axes_cm[1].set_xlabel('Predicted Class')
plt.tight_layout()
plt.show()

# 2. ROC & Precision-Recall Curves
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_results['y_proba'])
fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_results['y_proba'])
ax_roc.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {svm_results['roc_auc']:.4f})", color='#2563EB', lw=2)
ax_roc.plot(fpr_dt, tpr_dt, label=f"Decision Tree (AUC = {dt_results['roc_auc']:.4f})", color='#10B981', lw=2)
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance')
ax_roc.set_title('Receiver Operating Characteristic (ROC) Curves', fontweight='bold')
ax_roc.set_xlabel('False Positive Rate (1 - Specificity)')
ax_roc.set_ylabel('True Positive Rate (Recall)')
ax_roc.legend()

# PR Curves
prec_svm, rec_svm, _ = precision_recall_curve(y_test, svm_results['y_proba'])
prec_dt, rec_dt, _ = precision_recall_curve(y_test, dt_results['y_proba'])
ax_pr.plot(rec_svm, prec_svm, label=f"SVM (F1 = {svm_results['f1']:.4f})", color='#2563EB', lw=2)
ax_pr.plot(rec_dt, prec_dt, label=f"Decision Tree (F1 = {dt_results['f1']:.4f})", color='#10B981', lw=2)
ax_pr.axhline(y=y.mean(), color='gray', linestyle='--', label=f"Baseline ({y.mean():.2%})")
ax_pr.set_title('Precision-Recall Curves (Class-Imbalance Focus)', fontweight='bold')
ax_pr.set_xlabel('Recall (Sensitivity)')
ax_pr.set_ylabel('Precision (Positive Predictive Value)')
ax_pr.legend()

plt.tight_layout()
plt.show()


### Cell 10: Feature Importance Analysis
- **Decision Tree**: Native Gini Impurity Reduction
- **SVM**: Permutation Feature Importance (mean drop in test F1 score)


In [ ]:
from sklearn.inspection import permutation_importance

# 1. Decision Tree Gini Importance
dt_clf = best_dt.named_steps['classifier']
prep_dt = best_dt.named_steps['preprocessor']
feat_names = prep_dt.get_feature_names_out()

dt_importance = pd.DataFrame({
    'Feature': [f.replace('num__', '').replace('cat__', '').replace('bool__', '') for f in feat_names],
    'Importance': dt_clf.feature_importances_
}).sort_values('Importance', ascending=False)

# 2. SVM Permutation Importance
print("Calculating Permutation Importance for SVM on test set (n_repeats=5)...")
perm_svm = permutation_importance(best_svm, X_test, y_test, scoring='f1', n_repeats=5, random_state=42, n_jobs=-1)
svm_importance = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance_Mean_F1_Drop': perm_svm.importances_mean,
    'Importance_Std': perm_svm.importances_std
}).sort_values('Importance_Mean_F1_Drop', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=svm_importance.head(10), x='Importance_Mean_F1_Drop', y='Feature', ax=axes[0], color='#2563EB')
axes[0].set_title('Top 10 SVM Permutation Importance (Test F1 Drop)', fontweight='bold')
axes[0].set_xlabel('Mean F1 Score Reduction')

sns.barplot(data=dt_importance.head(10), x='Importance', y='Feature', ax=axes[1], color='#10B981')
axes[1].set_title('Top 10 Decision Tree Gini Feature Importance', fontweight='bold')
axes[1].set_xlabel('Relative Gini Impurity Reduction')

plt.tight_layout()
plt.show()


### Cell 11: Final Quantitative Comparison Table & Trade-offs


In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': 'SVM (Primary)',
        'Accuracy': round(svm_results['accuracy'], 4),
        'Precision': round(svm_results['precision'], 4),
        'Recall': round(svm_results['recall'], 4),
        'F1 Score': round(svm_results['f1'], 4),
        'ROC-AUC': round(svm_results['roc_auc'], 4),
        'CV Mean F1': round(svm_grid.best_score_, 4)
    },
    {
        'Model': 'Decision Tree (Comparison)',
        'Accuracy': round(dt_results['accuracy'], 4),
        'Precision': round(dt_results['precision'], 4),
        'Recall': round(dt_results['recall'], 4),
        'F1 Score': round(dt_results['f1'], 4),
        'ROC-AUC': round(dt_results['roc_auc'], 4),
        'CV Mean F1': round(dt_grid.best_score_, 4)
    }
])

print("=== FINAL QUANTITATIVE MODEL COMPARISON ===")
display(comparison_df)

best_model = "SVM" if svm_results['f1'] >= dt_results['f1'] else "Decision Tree"
print(f"\nBest-performing model by default F1-score metric: {best_model}")
print("\nAcademic Trade-off Note:")
print("- SVM achieves higher precision (57.59% vs 50.98%) and higher overall F1 (0.6419 vs 0.6285), minimizing false alarms.")
print("- Decision Tree captures higher recall (81.94% vs 72.51%), detecting more prospective buyers at the expense of 97 additional false alarms.")


### Cell 12: Interactive Inference Simulation (`predict_proba`)
Allows passing any session vector to compute predicted purchase probability and label.


In [ ]:
def predict_purchase_intention(session_dict, model_choice='SVM'):
    input_df = pd.DataFrame([session_dict])
    model = best_svm if model_choice == 'SVM' else best_dt
    
    proba = model.predict_proba(input_df)[0][1]
    prediction = int(model.predict(input_df)[0])
    label = "Purchase (True)" if prediction == 1 else "No Purchase (False)"
    
    print(f"--- PREDICTION RESULT ({model_choice}) ---")
    print(f"Outcome: {label}")
    print(f"Estimated Purchase Probability: {proba * 100:.2f}%")
    print(f"Disclaimer: This represents the model's posterior probability (predict_proba), not guaranteed customer action.\n")

# Example 1: High-Intent Buyer
sample_buyer = {
    'Administrative': 3, 'Administrative_Duration': 85.0,
    'Informational': 1, 'Informational_Duration': 42.0,
    'ProductRelated': 32, 'ProductRelated_Duration': 1250.0,
    'BounceRates': 0.005, 'ExitRates': 0.015, 'PageValues': 24.5,
    'SpecialDay': 0.0, 'Month': 'Nov', 'OperatingSystems': 2,
    'Browser': 2, 'Region': 1, 'TrafficType': 2,
    'VisitorType': 'Returning_Visitor', 'Weekend': 1
}

# Example 2: Immediate Bouncer
sample_bouncer = {
    'Administrative': 0, 'Administrative_Duration': 0.0,
    'Informational': 0, 'Informational_Duration': 0.0,
    'ProductRelated': 1, 'ProductRelated_Duration': 0.0,
    'BounceRates': 0.20, 'ExitRates': 0.20, 'PageValues': 0.0,
    'SpecialDay': 0.0, 'Month': 'Feb', 'OperatingSystems': 1,
    'Browser': 1, 'Region': 3, 'TrafficType': 1,
    'VisitorType': 'Returning_Visitor', 'Weekend': 0
}

predict_purchase_intention(sample_buyer, model_choice='SVM')
predict_purchase_intention(sample_bouncer, model_choice='SVM')
